# 13주차 랩 · 우리 조의 네트워크 분석 — 조별 발표

조별 실습 · 수업 시간에 끝까지 하고 발표한다

## 🎯 이번 랩의 질문

교재에서는 피렌체 가문의 혼인 네트워크로 Medici가 어떤 위치에 있었는지 보았다. 같은 방법을 조가 고른 실제 자료에 적용한다.

> **우리 조가 고른 네트워크에서 가장 중요한 node는 누구이고, 네트워크는 어떤 그룹으로 나뉘어 있나?**

### 진행 방법

- 4개 조가 아래 자료 가운데 하나를 골라 분석한다. 같은 자료를 고른 조는 범위(팀, 기간, 종목 등)를 다르게 잡는다.
- 자료를 받고 다듬는 일부터 조가 직접 한다.
- 지표는 교재와 같은 순서로 계산한다.
- 마지막 절의 발표 문항에 차례대로 답해 조별로 발표한다.
- 코드는 Colab에서 쓴다. AI가 필요하면 BA 스터디 도우미(https://balecture.tailcce2b.ts.net:8443)를 쓴다.


---
## 1. 자료 고르기

### 축구 경기의 패스

- **사이트**: StatsBomb Open Data (https://github.com/statsbomb/open-data)
- **자료**: 경기마다 모든 사건이 기록된 JSON 파일이 있다. 패스 기록에는 패스한 선수와 받은 선수가 적혀 있다.
- **node·link**: node는 선수, link는 두 선수 사이의 패스, 가중치는 패스 수
- **예**: 2022 카타르 월드컵에서 한 팀이 치른 모든 경기

**수집 방법**

1. 저장소의 `data` 폴더에 파일이 세 단계로 있다. 대회 목록 `competitions.json` → 대회·시즌별 경기 목록 `matches/대회번호/시즌번호.json` → 경기별 사건 기록 `events/경기번호.json`.
2. `competitions.json`에서 대회 이름이 FIFA World Cup이고 시즌이 2022인 행의 `competition_id`(43)와 `season_id`(106)를 찾는다.
3. 그 번호의 경기 목록에서 홈 팀이나 원정 팀이 우리 팀인 경기의 `match_id`를 모은다.
4. 경기마다 사건 기록을 읽어 하나로 합친다.

**힌트**

- 파일은 브라우저에서 열어 생김새를 먼저 본다. Colab에서는 GitHub 화면 주소가 아니라 원본 파일 주소(`https://raw.githubusercontent.com/statsbomb/open-data/master/data/...`)로 읽는다. `requests`로 받아 `.json()`을 붙이면 파이썬의 목록이 된다.
- 사건 하나는 안에 사전이 겹쳐 있다. `pd.json_normalize`로 펼치면 `type.name`, `team.name`, `player.name`, `pass.recipient.name`, `pass.outcome.name` 같은 열이 생긴다.
- `type.name`이 Pass이고 `team.name`이 우리 팀인 행만 남긴다. `pass.outcome.name`이 비어 있는 패스가 받는 선수에게 닿은 패스이다.
- 선수는 이름 전체로 구분한다. 한국 선수는 성이 같은 경우가 많다(김민재·김영권·김진수).
- 교재처럼 방향 없는 네트워크로 만들려면 A가 B에게 준 패스와 B가 A에게 준 패스를 같은 쌍으로 센다. 두 이름을 정렬해 한 쌍으로 만든 뒤 쌍마다 행 수를 세면 된다.

### 한국 영화 배우의 공동 출연

- **사이트**: 위키데이터 질의 서비스 (https://query.wikidata.org)
- **자료**: 아래 질의를 붙여 넣고 실행한 뒤 결과를 CSV로 내려받는다. 한 행이 영화 한 편과 그 영화에 출연한 배우 한 명이다.
- **node·link**: node는 배우, link는 같은 영화에 함께 출연한 관계, 가중치는 함께 출연한 영화 수
- **예**: 2000년 이후 한국 영화 (질의의 연도를 바꿔 범위를 정할 수 있다)

```
SELECT ?film ?filmLabel ?actorLabel WHERE {
  ?film wdt:P31 wd:Q11424; wdt:P495 wd:Q884; wdt:P577 ?date; wdt:P161 ?actor.
  FILTER(YEAR(?date) >= 2000)
  SERVICE wikibase:label { bd:serviceParam wikibase:language "ko,en". }
}
```

**수집 방법**

1. 질의 서비스 화면 왼쪽 편집 칸에 위 질의를 붙여 넣고 실행 단추(▶)를 누른다.
2. 결과 표 위의 다운로드 메뉴에서 CSV 파일을 고른다.
3. 받은 CSV를 Colab 왼쪽 파일 창에 올려 읽는다.

**힌트**

- 질의는 60초가 넘으면 멈춘다. 멈추면 연도 범위를 좁힌다.
- 질의의 뜻: `wdt:P31 wd:Q11424`는 영화, `wdt:P495 wd:Q884`는 제작 국가가 대한민국, `wdt:P577`은 개봉일, `wdt:P161`은 출연 배우이다.
- 같은 이름의 배우가 둘 이상일 수 있다. 첫 줄 SELECT에 `?actor`를 더하면 배우마다 고유 번호(Q로 시작)도 받는다.
- 개봉일이 여러 개인 영화는 같은 행이 되풀이되므로 중복 행을 먼저 지운다. 이름이 Q숫자로 나오는 배우는 한국어·영어 이름이 등록되지 않은 사람이다.
- link 목록은 영화마다 출연 배우를 두 명씩 짝지어 만든다(`itertools.combinations`). 짝마다 행 수를 세면 함께 출연한 영화 수가 된다.

### 코스피 종목의 주가

- **사이트**: FinanceDataReader (https://github.com/FinanceData/FinanceDataReader), Colab에서 `pip install finance-datareader`로 설치
- **자료**: 코스피 종목 목록(시가총액 포함)과 종목별 일별 주가
- **node·link**: node는 종목, link는 두 종목의 일별 수익률 상관계수가 조가 정한 기준 이상인 관계, 가중치는 상관계수
- **예**: 코스피 시가총액 상위 80개 종목의 2025년 일별 종가

**수집 방법**

1. 종목 목록 기능(`StockListing`)에 시장 이름 KOSPI를 넣어 종목 코드·이름·시가총액이 든 표를 받는다.
2. 시가총액 순으로 정렬해 분석할 종목을 고른다.
3. 주가 기능(`DataReader`)에 종목 코드와 시작일·종료일을 넣어 종목마다 일별 주가를 받는다.

**힌트**

- 종목 코드는 `005930`처럼 앞자리 0까지 여섯 자리 문자이다.
- 종목마다 받은 표에서 종가(`Close`) 열만 꺼내 날짜를 행, 종목을 열로 하는 표 하나로 모은다(`pd.concat`의 `axis=1`).
- 일별 수익률은 전날 대비 변화율(`pct_change`), 종목끼리의 상관계수 표는 `corr`로 구한다.
- 상관계수 표(종목 × 종목)를 link 목록으로 바꿀 때는 `stack`으로 한 줄씩 펼친 뒤 자기 자신과의 쌍과 순서만 바뀐 같은 쌍을 지운다.
- 종목이 많으면 받는 데 시간이 걸린다. 한 번 받은 표는 CSV로 저장해 두고 다시 읽는다.

---
## 2. 자료 받기와 네트워크 만들기

1. 고른 사이트에서 자료를 받아 Colab에서 읽는다.
2. 자료를 link 목록으로 만든다. 한 행이 두 node와 가중치다.
3. 가중치를 보고 어느 정도 이상인 관계만 link로 남길지 조가 정한다. 예: 패스 3회 이상, 함께 출연 2편 이상, 상관계수 0.5 이상
4. link 목록을 networkx 네트워크로 만들고 node 수와 link 수를 확인한다.
5. 네트워크 그림을 그린다.


In [ ]:
# 2. 자료 받기와 네트워크 만들기


---
## 3. 지표 계산

교재와 같은 순서로 계산한다.

1. **degree centrality**: node마다 직접 이어진 link의 수
2. **weighted degree**: node마다 이어진 link의 가중치 합. 축구는 주고받은 패스 수, 영화는 함께 출연한 영화 수, 주가는 상관계수의 합이다
3. **betweenness centrality**: node마다 다른 두 node 사이의 최단 경로가 그 node를 지나는 정도. 교재와 같이 나누지 않은 값으로 구한다
4. **robustness**: node를 하나씩 빼며 node마다 $r_i$ = (남은 node 수) − (가장 큰 connected component의 node 수)
5. **community**: networkx의 community 찾기 기능으로 그룹을 나누고, 그룹마다 modularity $q_c = L_c/m - (d_c/2m)^2$
6. **그룹 사이 link**: 양 끝이 서로 다른 그룹에 있는 link와 그 수

**가중치를 넣어 계산하기** — 교재의 networkx 함수는 모두 가중치를 받을 수 있다.

- link 목록을 네트워크로 만들 때 가중치 열도 link에 붙인다(`from_pandas_edgelist`의 `edge_attr`).
- **weighted degree**: `degree`에 가중치 열 이름을 알려 주면 된다.
- **betweenness centrality**: 이 함수는 가중치를 거리로 읽는다. 패스가 많을수록 가까운 사이이므로, 1/가중치로 거리 열을 만들어 그 열을 넣는다.
- **community**: `greedy_modularity_communities`는 가중치를 따로 알려 주지 않으면 쓰지 않는다. $q_c$의 $L_c$, $d_c$, $m$도 link 수 대신 가중치 합으로 구한다.
- **robustness**: 가중치와 상관없으므로 교재 그대로 계산한다.

In [ ]:
# 3. 지표 계산


---
## 4. 발표

수업 시간에 조별로 발표한다. 아래 문항에 차례대로 답한다. 숫자는 코드로 구한 값을 쓴다.

1. **자료**: 어느 사이트에서 무엇을 받았나? node와 link는 무엇이고, 어떤 기준으로 link를 남겼나? node와 link는 몇 개인가?
2. **degree centrality와 weighted degree**: 각각 1\~3위 node는 누구인가? 왜 이 node들이 많은 node와 이어져 있나? 두 순위가 다르면 누가 왜 다른가?
3. **betweenness centrality**: 1\~3위 node는 누구인가? degree centrality 순위와 다른 node가 있다면 누구이고, 왜 그런가?
4. **robustness**: $r_i$가 가장 큰 node는 누구인가? 그 node가 빠지면 어떤 node가 떨어져 나가나?
5. **community**: 그룹은 몇 개이고, 그룹마다 $q_c$는 얼마인가? 각 그룹의 node들은 무엇이 같은가? (포지션, 장르·감독, 업종 등)
6. **그룹 사이**: 어느 그룹 사이가 약하게 이어져 있나? 그룹 사이를 잇는 node는 누구인가?
7. **결론**: 이 네트워크에서 가장 중요한 node는 누구이고, 그 근거는 무엇인가? 이 결과로 어떤 결정을 내릴 수 있나? (예: 감독의 전술, 영화 제작사의 캐스팅, 투자자의 분산투자)